# Gymnasium Environments with PPO

## Setup
We will use gymnasium and torch for this implementation.

In [ ]:
ENV_ID = "CartPole-v1"
#ENV_ID = "LunarLander-v3"
ALGO_ID = "REINFORCE"  # Change to "REINFORCE" for REINFORCE algorithm
#ALGO_ID = "PPO"

Install packages:

In [ ]:
#%CXX=clang++ pip install "gymnasium[box2d]" --quiet
%pip install stable-baselines3 wandb swig tsilva-notebook-utils==0.0.121 --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*")

Load secrets:

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

Retrieve enviroment variables:

In [ ]:
import torch
import numpy as np
from tsilva_notebook_utils.torch import get_default_device

DEVICE = get_default_device()
DEVICE

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed
from dataclasses import dataclass
from typing import Union, Tuple

@dataclass
class RLConfig:
    # Environment
    env_id: str
    seed: int = 42
    
    # Training
    max_epochs: int = -1
    gamma: float = 0.99
    lam: float = 0.95
    clip_epsilon: float = 0.2
    batch_size: int = 64
    train_rollout_steps: int = 2048
    
    # Evaluation
    eval_interval: int = 10
    eval_episodes: int = 32
    reward_threshold: float = 200
    
    # Networks
    policy_lr: float = 3e-4
    value_lr: float = 1e-3
    hidden_dim: Union[int, Tuple[int, ...]] = 64
    entropy_coef: float = 0.01
    
    # Other
    normalize: bool = False
    mean_reward_window: int = 100
    rollout_interval: int = 10
    n_envs: Union[str, int] = "auto"
    async_rollouts: bool = True
    
    # Environment-specific configurations with flat structure
    # Use these as reference: https://github.com/DLR-RM/rl-baselines3-zoo/tree/master/hyperparams
    ENV_CONFIGS = {
        "CartPole-v1": {
            # Default config for this environment (applies to all algorithms unless overridden)
            "default": dict(
                train_rollout_steps=512,
                batch_size=256,
                rollout_interval=1,
                eval_interval=20,
                eval_episodes=5,
                reward_threshold=475,
                policy_lr=1e-3,
                value_lr=1e-3,
                hidden_dim=32,
            ),
            # Algorithm-specific overrides
            "reinforce": dict(
                train_rollout_steps=2048,
                batch_size=512,
                policy_lr=1e-3,
                entropy_coef=0.02,
            ),
        },
        "Acrobot-v1": {
            "default": dict(
                gamma=0.99,
                lam=0.98,
                clip_epsilon=0.2,
                batch_size=128,
                train_rollout_steps=2048,
                eval_interval=5,
                reward_threshold=-100,
                policy_lr=3e-4,
                value_lr=3e-4,
                hidden_dim=(128, 64),
                entropy_coef=0.01,
                rollout_interval=1
            ),
            "reinforce": dict(
                train_rollout_steps=4096,
                batch_size=256,
                policy_lr=5e-4,
                entropy_coef=0.05,
            ),
        },
        # TODO:
        #n_envs: 16
        #n_epochs: 4 
        #n_steps: 1024
        "LunarLander-v3": {
            "default": dict(
                reward_threshold=200,
                total_timesteps=1e6, # TODO: call this n_timesteps
                gamma=0.999,
                # TODO: this is not being propagated to collect_rollouts
                lam=0.98, # gae_lambda: 0.98
                clip_epsilon=0.2,
                batch_size=64,
                eval_interval=2,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=32,
                entropy_coef=0.01
            ),
            "reinforce": dict(
                entropy_coef=0.03,
                batch_size=128,
            ),
        },
        "Pendulum-v1": {
            "default": dict(
                gamma=0.99,
                lam=0.95,
                clip_epsilon=0.2,
                batch_size=64,
                eval_interval=2,
                eval_episodes=5,
                reward_threshold=-200,
                policy_lr=3e-4,
                value_lr=1e-3,
                hidden_dim=(128, 64),
                entropy_coef=0.0
            ),
            "reinforce": dict(
                entropy_coef=0.02,
                batch_size=128,
            ),
        },
        "MountainCar-v0": {
            "default": dict(
                gamma=0.99,
                lam=0.97,
                clip_epsilon=0.15,
                batch_size=16,
                eval_interval=2,
                eval_episodes=10,
                reward_threshold=-110,
                policy_lr=1e-4,
                value_lr=5e-4,
                hidden_dim=(128, 64),
                entropy_coef=0.05
            ),
            "reinforce": dict(
                entropy_coef=0.08,
                batch_size=32,
            ),
        },
    }
    
    @classmethod
    def create(cls, env_id: str, algorithm: str = "ppo") -> 'RLConfig':
        """
        Create config with hierarchical overrides:
        1. Start with dataclass defaults
        2. Apply environment default config
        3. Apply algorithm-specific config for that environment
        """
        config = cls(env_id=env_id)
        
        # Level 2 & 3: Apply environment and algorithm configs
        if env_id in cls.ENV_CONFIGS:
            env_config = cls.ENV_CONFIGS[env_id]
            
            # Apply environment default config first
            if "default" in env_config:
                for key, value in env_config["default"].items():
                    setattr(config, key, value)
            
            # Apply algorithm-specific config if it exists
            if algorithm in env_config:
                for key, value in env_config[algorithm].items():
                    setattr(config, key, value)
        
        return config

# Create configs for different algorithms
CONFIG = RLConfig.create(ENV_ID, ALGO_ID)
CONFIG

Build environment:

In [ ]:
from tsilva_notebook_utils.gymnasium import log_env_info

# Set random seed for reproducibility
set_random_seed(CONFIG.seed)

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG.env_id, 
    norm_obs=CONFIG.normalize, 
    n_envs=n_envs if n_envs is not None else CONFIG.n_envs, 
    seed=seed
)

# Test building env
env = build_env(CONFIG.seed)
log_env_info(env)

## Build Agent

Define models:

In [ ]:
class MLPNet(nn.Module):
    """Reusable MLP with configurable hidden dimensions"""
    
    def __init__(self, input_dim, output_dim, hidden_dim=64, activation=nn.ReLU):
        super().__init__()
        
        if isinstance(hidden_dim, (int, float)):
            hidden_dims = [int(hidden_dim)]
        else:
            hidden_dims = [int(dim) for dim in hidden_dim]
        
        layers = []
        current_dim = input_dim
        
        for hidden_size in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_size),
                activation()
            ])
            current_dim = hidden_size
        
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

class PolicyNet(MLPNet):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__(obs_dim, act_dim, hidden_dim)

class ValueNet(MLPNet):
    def __init__(self, obs_dim, hidden_dim=64):
        super().__init__(obs_dim, 1, hidden_dim)

Define agent class:

In [ ]:
import time
import multiprocessing
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from tsilva_notebook_utils.gymnasium import (
    collect_rollouts, group_trajectories_by_episode,
    RolloutDataset, MetricTracker, AsyncRolloutCollector, SyncRolloutCollector
)

# ---------------------------------------------------------------------
class Agent(pl.LightningModule):
    """Base agent class with common RL functionality"""
    
    def __init__(self, obs_dim, act_dim, config, build_env_fn):
        super().__init__()
        
        # Store core attributes
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.config = config
        self.build_env_fn = build_env_fn
        
        # Common RL components
        self.env = build_env_fn(config.seed)
        self.metrics = MetricTracker(self)
        self.rollout_ds = RolloutDataset()
        self.episode_reward_deque = deque(maxlen=config.mean_reward_window)
        
        # Rollout collection
        rollout_collector_cls = AsyncRolloutCollector if config.async_rollouts else SyncRolloutCollector
        self.rollout_collector = rollout_collector_cls(build_env_fn, config, obs_dim, act_dim)
        
        # Training state
        self.automatic_optimization = False
        self.training_start_time = None
        self.total_steps = 0  # Track total training steps consumed
        
    def create_models(self):
        """Override in subclass to create algorithm-specific models"""
        raise NotImplementedError("Subclass must implement create_models()")
        
    def compute_loss(self, batch):
        """Override in subclass to compute algorithm-specific loss"""
        raise NotImplementedError("Subclass must implement compute_loss()")
        
    def optimize_models(self, loss_results):
        """Override in subclass to implement algorithm-specific optimization"""
        raise NotImplementedError("Subclass must implement optimize_models()")
        
    def get_models_for_rollout(self):
        """Override in subclass to return models needed for rollout collection"""
        raise NotImplementedError("Subclass must implement get_models_for_rollout()")

    def setup(self, stage: str):
        if stage == "fit":
            policy_model, value_model = self.get_models_for_rollout()
            self.rollout_collector.initialize_with_models(policy_model, value_model)
            self.rollout_collector.start()
            
            print("Waiting for initial rollout...")
            while True:
                if self.rollout_collector.is_ready_for_initial_rollout():
                    trajectories = self.rollout_collector.get_rollout(timeout=2.0)
                    if trajectories is not None:
                        self._update_rollout_data(trajectories)
                        break
                print("Still waiting for rollout...")

    def train_dataloader(self):
        return DataLoader(
            self.rollout_ds,
            batch_size=self.config.batch_size,
            shuffle=True,
            # Pin memory is not supported on MPS
            pin_memory=True if self.device.type != 'mps' else False,
            # TODO: Persistent workers + num_workers is fast but doesn't converge
            persistent_workers=True if self.device.type != 'mps' else False,
            # Using multiple workers stalls the start of each epoch when persistent workers are disabled
            num_workers=multiprocessing.cpu_count() // 2 if self.device.type != 'mps' else 0
        )

    def on_fit_start(self):
        self.training_start_time = time.time()
        self.total_steps = 0  # Reset step counter
        mode = "async" if self.config.async_rollouts else "sync"
        print(f"Training started in {mode} mode at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    def on_fit_end(self):
        self.rollout_collector.stop()
        if self.training_start_time:
            total_time = time.time() - self.training_start_time
            print(f"Training completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def on_train_epoch_start(self):
        self.metrics.reset()
        policy_model, value_model = self.get_models_for_rollout()
        self.rollout_collector.update_models(
            policy_model.state_dict(), value_model.state_dict()
        )
        
        # Collect new rollout if needed
        if (self.current_epoch + 1) % self.config.rollout_interval == 0:
            self._collect_and_update_rollout()

    def on_train_epoch_end(self):
        # Log epoch metrics
        epoch_metrics = self.metrics.compute_epoch_means()
        if epoch_metrics:
            self.metrics.log_metrics(epoch_metrics, prefix="epoch")
        
        # Evaluation
        if (self.current_epoch + 1) % self.config.eval_interval == 0:
            self._evaluate_and_check_stopping()

    def training_step(self, batch, batch_idx):
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch

        # Track total steps consumed by trainer
        batch_size = states.size(0)
        self.total_steps += batch_size

        # Compute algorithm-specific losses and metrics
        loss_results = self.compute_loss(batch)

        # Track step metrics
        self.metrics.add_step_metrics(loss_results)

        # Optimize models
        self.optimize_models(loss_results)

        # Log training metrics
        self._log_training_metrics(loss_results, advantages, values, returns)

        return sum(loss for key, loss in loss_results.items() if 'loss' in key)

    def _collect_and_update_rollout(self):
        """Collect and update rollout data"""
        timeout = 2.0 if self.config.async_rollouts else 1.0
        trajectories = self.rollout_collector.get_rollout(timeout=timeout)
        
        if trajectories is not None:
            self._update_rollout_data(trajectories)
            self.metrics.log_single('rollout/queue_updated', 1.0)
        else:
            self.metrics.log_single('rollout/queue_miss', 1.0)
    
    def _update_rollout_data(self, trajectories):
        """Update rollout dataset and episode rewards"""
        self.rollout_ds.update(*trajectories)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards:
            self.episode_reward_deque.append(float(r))

    def _log_training_metrics(self, loss_results, advantages, values, returns):
        """Log common training metrics"""
        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) > 0 else 0
        
        # Core metrics that most algorithms will have
        train_metrics = {
            'mean_reward': mean_reward,
            'total_steps': self.total_steps,
        }
        
        # Add algorithm-specific loss metrics
        for key, value in loss_results.items():
            if 'loss' in key or key in ['entropy', 'kl_divergence', 'explained_variance']:
                train_metrics[key] = value
        
        additional_metrics = {
            'advantage_mean': advantages.mean(),
            'advantage_std': advantages.std(),
            'value_mean': values.mean(),
            'returns_mean': returns.mean(),
        }
        
        # Add any additional algorithm-specific metrics
        for key, value in loss_results.items():
            if key not in train_metrics and key not in additional_metrics:
                additional_metrics[key] = value
        
        self.metrics.log_metrics(train_metrics, prefix="train", prog_bar=True)
        self.metrics.log_metrics(additional_metrics, prefix="train", prog_bar=False)

    def _evaluate_and_check_stopping(self):
        """Evaluate model and check for early stopping"""
        eval_seed = np.random.randint(0, 1_000_000)
        eval_env = self.build_env_fn(eval_seed)
        
        policy_model, value_model = self.get_models_for_rollout()
        policy_model.eval()
        try:
            eval_mean_reward = self._run_evaluation(eval_env, policy_model, value_model)
            self.metrics.log_single('eval/mean_reward', eval_mean_reward, prog_bar=True)
            
            if eval_mean_reward >= self.config.reward_threshold:
                print(f"Early stopping at epoch {self.current_epoch} with eval mean reward {eval_mean_reward:.2f} >= threshold {self.config.reward_threshold}")
                self.trainer.should_stop = True
                
        finally:
            policy_model.train()
            eval_env.close()

    def _run_evaluation(self, env, policy_model, value_model):
        """Run evaluation and log rollout metrics"""
        start = time.time()
        trajectories, _ = collect_rollouts(
            env, policy_model, value_model,
            n_episodes=self.config.eval_episodes, deterministic=False
        )
        elapsed = time.time() - start

        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        mean_episode_reward = np.mean(episode_rewards)
        
        # Log rollout metrics
        rollout_metrics = {
            'mean_reward': mean_episode_reward,
            'num_episodes': len(episodes),
            'num_steps': len(trajectories[0]),
            'avg_steps_per_episode': len(trajectories[0]) / (len(episodes) + 1e-3),
            'time_elapsed': elapsed,
            'steps_per_second': len(trajectories[0]) / (elapsed + 1e-3)
        }
        
        self.metrics.log_metrics(rollout_metrics, prefix="rollout")
        return mean_episode_reward


In [ ]:
class REINFORCELoss:
    def __init__(self, entropy_coef):
        self.entropy_coef = entropy_coef
    
    def compute(self, states, actions, returns, policy_model):
        # Policy loss using REINFORCE (policy gradient with Monte Carlo returns)
        logits = policy_model(states)
        dist = Categorical(logits=logits)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy().mean()
        
        # REINFORCE loss: -log_prob * return (negative because we want to maximize)
        policy_loss = -(log_probs * returns).mean() - self.entropy_coef * entropy
        
        # Metrics
        # TODO: we should probably return the actual values? (make sure they are detached?)
        return {
            'policy_loss': policy_loss,
            'entropy': entropy,
            'log_prob_mean': log_probs.mean(),
            'returns_mean': returns.mean()
        }

In [ ]:
class REINFORCEAgent(Agent):
    """REINFORCE-specific agent implementation"""
    
    def __init__(self, obs_dim, act_dim, config, build_env_fn):
        super().__init__(obs_dim, act_dim, config, build_env_fn)
        self.save_hyperparameters(ignore=['build_env_fn'])
        
        # Create REINFORCE-specific models and components
        self.create_models()
        self.reinforce_loss = REINFORCELoss(config.entropy_coef)
        
    def create_models(self):
        """Create REINFORCE-specific policy model (no value model needed)"""
        self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config.hidden_dim)
        # REINFORCE doesn't use a value function for training
        self.value_model = None
        
    def get_models_for_rollout(self):
        """Return models needed for rollout collection"""
        # REINFORCE only needs policy model for rollouts
        return self.policy_model, None
        
    def compute_loss(self, batch):
        """Compute REINFORCE-specific losses"""
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch
        
        # REINFORCE uses Monte Carlo returns directly (not advantages)
        return self.reinforce_loss.compute(
            states, actions, returns, self.policy_model
        )
        
    def optimize_models(self, loss_results):
        """Optimize REINFORCE policy model"""
        optimizer = self.optimizers()
        
        # Optimize policy
        optimizer.zero_grad()
        self.manual_backward(loss_results['policy_loss'])
        optimizer.step()

    def configure_optimizers(self):
        # REINFORCE only needs policy optimizer
        return torch.optim.Adam(self.policy_model.parameters(), lr=self.config.policy_lr)

    def forward(self, x):
        return self.policy_model(x)

In [ ]:
from torch.distributions import Categorical

class PPOLoss:
    def __init__(self, clip_epsilon, entropy_coef):
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
    
    def compute(self, states, actions, old_logps, advantages, returns, policy_model, value_model):
        # Policy loss
        logits = policy_model(states)
        dist = Categorical(logits=logits)
        new_logps = dist.log_prob(actions)
        
        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * advantages
        entropy = dist.entropy().mean()
        
        policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy
        
        # Value loss
        value_pred = value_model(states).squeeze()
        value_loss = 0.5 * ((returns - value_pred) ** 2).mean()
        
        # Metrics
        clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
        kl_div = (old_logps - new_logps).mean()
        approx_kl = ((ratio - 1) - torch.log(ratio)).mean()
        explained_var = 1 - torch.var(returns - value_pred) / torch.var(returns)
        
        return {
            'policy_loss': policy_loss,
            'value_loss': value_loss,
            'entropy': entropy,
            'clip_fraction': clip_fraction,
            'kl_div': kl_div,
            'approx_kl': approx_kl,
            'explained_var': explained_var
        }

In [ ]:
class PPOAgent(Agent):
    """PPO-specific agent implementation"""
    
    def __init__(self, obs_dim, act_dim, config, build_env_fn):
        super().__init__(obs_dim, act_dim, config, build_env_fn)
        self.save_hyperparameters(ignore=['build_env_fn'])
        
        # Create PPO-specific models and components
        self.create_models()
        self.ppo_loss = PPOLoss(config.clip_epsilon, config.entropy_coef)
        
    def create_models(self):
        """Create PPO-specific policy and value models"""
        self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config.hidden_dim)
        self.value_model = ValueNet(self.obs_dim, self.config.hidden_dim)
        
    def get_models_for_rollout(self):
        """Return models needed for rollout collection"""
        return self.policy_model, self.value_model
        
    def compute_loss(self, batch):
        """Compute PPO-specific losses"""
        states, actions, rewards, dones, old_logps, values, advantages, returns, frames = batch
        
        return self.ppo_loss.compute(
            states, actions, old_logps, advantages, returns, 
            self.policy_model, self.value_model
        )
        
    def optimize_models(self, loss_results):
        """Optimize PPO policy and value models"""
        opt_policy, opt_value = self.optimizers()
        
        # Optimize policy
        opt_policy.zero_grad()
        self.manual_backward(loss_results['policy_loss'])
        opt_policy.step()

        # Optimize value function
        opt_value.zero_grad()
        self.manual_backward(loss_results['value_loss'])
        opt_value.step()

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.config.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.config.value_lr)
        ]

    def forward(self, x):
        return self.policy_model(x)


In [ ]:

import wandb
from pytorch_lightning.loggers import WandbLogger
from tsilva_notebook_utils.lightning import WandbCleanup

# Create PPO agent
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n if hasattr(env.action_space, 'n') else env.action_space.shape[0]
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG, build_env)

agent_cls = None
if ALGO_ID.upper() == "PPO": agent_cls = PPOAgent
elif ALGO_ID.upper() == "REINFORCE": agent_cls = REINFORCEAgent
else: raise ValueError(f"Unsupported algorithm: {ALGO_ID}. Choose 'PPO' or 'REINFORCE'")
agent = agent_cls(obs_dim, act_dim, CONFIG, build_env)

wandb_logger = WandbLogger(
    project=f"{ENV_ID}",
    name=f"{ALGO_ID}-{wandb.util.generate_id()[:5]}",
    log_model=True
)

# Print W&B run URL explicitly
print(f"🔗 W&B Run: {wandb_logger.experiment.url}")

trainer = pl.Trainer(
    logger=wandb_logger,
    log_every_n_steps=10,
    max_epochs=CONFIG.max_epochs,
    enable_progress_bar=True,
    enable_checkpointing=False,  # Disable checkpointing for speed
    accelerator="auto",
    callbacks=[WandbCleanup()]
)

# Fit the model
trainer.fit(ppo_agent)

## Evaluate

In [ ]:
import random
from tsilva_notebook_utils.gymnasium import render_episode_frames

n_episodes = 8
trajectories, _ = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=n_episodes),
    ppo_agent.policy_model,
    n_episodes=n_episodes,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))

Based on your PPO training code, here are all the metrics you're logging and how to monitor them:

## Core Performance Metrics (Primary Focus)

### **`train/mean_reward`** & **`eval/mean_reward`**
- **What**: Rolling average of episode rewards during training vs. evaluation performance
- **Monitor**: Primary success indicator - should trend upward toward your threshold (200 for LunarLander)
- **Action**: If plateauing, adjust learning rates or entropy coefficient

### **`train/total_steps`**
- **What**: Total environment steps consumed by the agent
- **Monitor**: Sample efficiency - fewer steps to reach threshold = better
- **Action**: Compare across hyperparameter settings to find most sample-efficient config

## Policy Learning Metrics

### **`epoch/policy_loss`**
- **What**: PPO clipped policy gradient loss
- **Monitor**: Should generally decrease but may fluctuate due to clipping
- **Action**: If consistently increasing, reduce learning rate or increase clip epsilon

### **`epoch/entropy`**
- **What**: Policy action distribution entropy (exploration measure)
- **Monitor**: Should start high and gradually decrease as policy becomes more deterministic
- **Action**: If drops too quickly, increase entropy coefficient; if too high, decrease it

### **`epoch/clip_fraction`**
- **What**: Fraction of policy updates that hit the PPO clipping bounds
- **Monitor**: Should be 0.1-0.3; higher means aggressive policy changes
- **Action**: If too high (>0.5), reduce learning rate or decrease clip epsilon

### **`epoch/kl_div`** & **`epoch/approx_kl`**
- **What**: KL divergence between old and new policies
- **Monitor**: Should be small (<0.1); large values indicate unstable training
- **Action**: If too high, reduce policy learning rate

## Value Function Metrics

### **`epoch/value_loss`**
- **What**: Mean squared error between predicted and actual returns
- **Monitor**: Should decrease over time as value function improves
- **Action**: If not decreasing, increase value learning rate or network capacity

### **`epoch/explained_var`**
- **What**: How well value function explains return variance (0-1 scale)
- **Monitor**: Should increase toward 1.0; >0.7 is good
- **Action**: If low, the value function isn't learning well - check value_lr or network size

### **`train/advantage_mean`** & **`train/advantage_std`**
- **What**: Statistics of computed advantages (GAE)
- **Monitor**: Mean should be near 0; std indicates advantage magnitude
- **Action**: If mean drifts from 0, value function may be biased

## Data Collection Metrics

### **`rollout/queue_updated`** vs **`rollout/queue_miss`**
- **What**: Success rate of async rollout collection
- **Monitor**: More updates than misses indicates healthy data pipeline
- **Action**: If many misses, increase queue size or reduce rollout interval

### **`rollout/steps_per_second`**
- **What**: Environment interaction speed
- **Monitor**: Higher is better for training efficiency
- **Action**: Use for comparing sync vs async modes

## Monitoring Strategy



In [ ]:
# Key metrics to watch on W&B dashboard:
primary_metrics = [
    "eval/mean_reward",           # Main success indicator
    "train/mean_reward",          # Training progress
    "epoch/explained_var",        # Value function quality
    "epoch/entropy",              # Exploration level
    "epoch/clip_fraction"         # Policy update stability
]

warning_conditions = {
    "epoch/clip_fraction > 0.5": "Reduce policy_lr or clip_epsilon",
    "epoch/approx_kl > 0.1": "Reduce policy_lr", 
    "epoch/explained_var < 0.3": "Increase value_lr or network size",
    "epoch/entropy < 0.01": "Increase entropy_coef",
    "rollout/queue_miss > rollout/queue_updated": "Check async collection"
}



## Red Flags to Watch For

1. **Reward plateau** with high clip_fraction → Reduce learning rates
2. **Value loss not decreasing** → Value function isn't learning properly
3. **Entropy drops to near 0** → Agent stopped exploring, increase entropy_coef
4. **Large KL divergence spikes** → Policy updates too aggressive
5. **Many rollout queue misses** → Data collection bottleneck

Focus primarily on `eval/mean_reward` trending toward your threshold, with `explained_var` and `entropy` as health checks for your learning process.